In [1]:
import pennylane as qml
from pennylane import numpy as np
from pennylane.optimize import NesterovMomentumOptimizer

In [23]:
import os
os.getcwd()

'/Users/beth/QuantumVentures/src/VQCs'

In [2]:
np.random.seed(499122177);

In [3]:
WIRES = 4
dev = qml.device('default.qubit', wires = WIRES);

In [4]:
def layer_prep(layer_weights):
    assert len(layer_weights) == WIRES;
    for qubit, wt in enumerate(layer_weights):
        qml.Rot(*wt, wires = qubit);

    # Entangling the Qubits
    for i in range(WIRES - 1):
        qml.CNOT([i, i + 1]);
    qml.CNOT([WIRES - 1, 0]);

In [5]:
def prepare_state(x):
    qml.BasisState(x, wires = [x for x in range(WIRES)]);

In [6]:
@qml.qnode(dev)
def ckt(weights, x):
    prepare_state(x);

    for layer_weights in weights:
        layer_prep(layer_weights);

    return qml.expval(qml.PauliZ(0));

In [7]:
def classifier(weights, bias, x):
    return ckt(weights, x) + bias

In [8]:
def sq_loss(labels, pred):
    return np.mean((labels - qml.math.stack(pred)) ** 2);

In [9]:
def cost(weights, bias, X, Y):
    pred = [classifier(weights, bias, x) for x in X];
    return sq_loss(Y, pred);

In [10]:
def acc(labels, pred):
    a = sum(abs(l - p) < 1e-5 for l, p in zip(labels, pred));
    a = a / len(labels);
    return a;

In [11]:
## Preparing the data
dataset = np.loadtxt('parity_train.txt', dtype = 'int')
X = dataset[:, :-1]
Y = dataset[:, -1] * 2 - 1

In [12]:
num_qubits = 4
num_layers = 2
weights = 0.01 * np.random.randn(num_layers, num_qubits, 3, requires_grad = True);
bias = np.array(0.0, requires_grad = True);

In [19]:
print(qml.draw(ckt, 'mpl')(weights, 15))

TypeError: 'in <string>' requires string as left operand, not int

In [14]:
opt = NesterovMomentumOptimizer(0.5);
batch_size = 5

In [15]:
wts = weights
bs = bias

for _ in range(100):

    batch_idx = np.random.randint(0, len(X), (batch_size, ));
    X_batch = X[batch_idx]
    Y_batch = Y[batch_idx]
    
    wts, bs = opt.step(cost, wts, bs, X = X_batch, Y = Y_batch);
    predictions = [np.sign(classifier(weights, bias, x)) for x in X]

    curr_cost = cost(wts, bs, X, Y);
    ac = acc(Y, predictions);
    print(f"Iter: {_+1:4d} | Cost: {curr_cost:0.7f} | Accuracy: {ac:0.7f}")

Iter:    1 | Cost: 1.9980396 | Accuracy: 0.5000000
Iter:    2 | Cost: 1.9748360 | Accuracy: 0.5000000
Iter:    3 | Cost: 1.8810026 | Accuracy: 0.5000000
Iter:    4 | Cost: 1.3833483 | Accuracy: 0.5000000
Iter:    5 | Cost: 1.6446684 | Accuracy: 0.5000000
Iter:    6 | Cost: 0.9423803 | Accuracy: 0.5000000
Iter:    7 | Cost: 1.4753731 | Accuracy: 0.5000000
Iter:    8 | Cost: 1.1151000 | Accuracy: 0.5000000
Iter:    9 | Cost: 1.0734451 | Accuracy: 0.5000000
Iter:   10 | Cost: 1.0680085 | Accuracy: 0.5000000
Iter:   11 | Cost: 1.0395162 | Accuracy: 0.5000000
Iter:   12 | Cost: 1.2992767 | Accuracy: 0.5000000
Iter:   13 | Cost: 1.0088474 | Accuracy: 0.5000000
Iter:   14 | Cost: 1.4773060 | Accuracy: 0.5000000
Iter:   15 | Cost: 1.1108816 | Accuracy: 0.5000000
Iter:   16 | Cost: 1.4232978 | Accuracy: 0.5000000
Iter:   17 | Cost: 0.2477509 | Accuracy: 0.5000000
Iter:   18 | Cost: 0.2009465 | Accuracy: 0.5000000
Iter:   19 | Cost: 0.3649717 | Accuracy: 0.5000000
Iter:   20 | Cost: 1.1760145 | 